In [ ]:
# ==========================================
# FORCE PYTORCH TO ONLY SEE 1 GPU
# ==========================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install onnx onnxscript onnxruntime matplotlib seaborn scikit-learn -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2
from PIL import Image, ImageFile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

ImageFile.LOAD_TRUNCATED_IMAGES = True

# 1. Configuration & Path
DATA_DIR = "/kaggle/input/datasets/mostafaabla/garbage-classification/garbage_classification"
BATCH_SIZE = 32
NUM_EPOCHS = 40      
LEARNING_RATE = 0.0001  
NUM_CLASSES = 2      

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# 2. Custom Dataset Class
class BinaryGarbageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
                
            if class_name.lower() in ['glass', 'trash', 'biological', 'clothes', 'shoes', 'battery']:
                continue
                
            label = 0 if class_name.lower() == 'plastic' else 1
            
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# 3. Augmentations 
train_transforms = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = v2.Compose([
    v2.Resize(size=(256, 256), antialias=True),
    v2.CenterCrop(size=(224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cutmix = v2.CutMix(num_classes=NUM_CLASSES)
mixup = v2.MixUp(num_classes=NUM_CLASSES)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

# 4. Data Loading & Splitting
full_dataset = BinaryGarbageDataset(DATA_DIR)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform = val_transforms

train_labels = [full_dataset.labels[i] for i in train_dataset.indices]
class_counts = np.bincount(train_labels)

class_weights_arr = 1.0 / class_counts
sample_weights = [class_weights_arr[label] for label in train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 5. Model Setup (ResNet-50)
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for param in model.parameters():
    param.requires_grad = True

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
nn.init.kaiming_normal_(model.fc.weight, mode='fan_out', nonlinearity='relu')
if model.fc.bias is not None:
    nn.init.zeros_(model.fc.bias)

model = model.to(device)

# 6. Loss, Optimizer, Scheduler
class UnweightedFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(UnweightedFocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

criterion = UnweightedFocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# 7. Training Loop (Pure FP32)
best_val_acc = 0.0
model_save_path = "/kaggle/working/best_resnet.pth"
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        if np.random.rand() > 0.5:
            inputs, labels = cutmix_or_mixup(inputs, labels)
            
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    scheduler.step()
    epoch_loss = running_loss / (len(train_loader) * BATCH_SIZE)
    history['train_loss'].append(epoch_loss)
    
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    val_epoch_loss = val_loss / val_size
    val_epoch_acc = val_corrects.double() / val_size
    
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc.item())
    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.4f}")
    
    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        torch.save(model.state_dict(), model_save_path)

# 8. Evaluation & ONNX Export
model.load_state_dict(torch.load(model_save_path))
model.eval()

all_preds, all_labels = [], []
PLASTIC_THRESHOLD = 0.35 

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)
        preds = torch.where(probs[:, 0] >= PLASTIC_THRESHOLD, torch.tensor(0).to(device), torch.tensor(1).to(device))
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nFINAL MODEL EVALUATION")
target_names = ['Plastic (0)', 'Non-Plastic (1)']
print(classification_report(all_labels, all_preds, target_names=target_names))

model.to("cpu") 
dummy_input = torch.randn(1, 3, 224, 224)
onnx_file_path = "/kaggle/working/plastic_sorter_resnet.onnx"
torch.onnx.export(
    model, dummy_input, onnx_file_path, export_params=True, opset_version=12,
    do_constant_folding=True, input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"ONNX Export successful: {onnx_file_path}")